<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set2/blob/main/DeepSeek_Set2_OneShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA NOT available. Using CPU.")

CUDA is available! Using GPU.
GPU device name: NVIDIA A100-SXM4-40GB


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            83Gi       6.6Gi        45Gi        20Mi        30Gi        76Gi
Swap:             0B          0B          0B


In [ ]:
from google.colab import files
import zipfile
import torch

import os
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
import shutil

# CLEANUP - Remove old folders before extraction
if os.path.exists('input_folder'):
    shutil.rmtree('input_folder')
if os.path.exists('output_folder'):
    shutil.rmtree('output_folder')


In [ ]:
import shutil
import os

# Delete EVERYTHING (folders AND old zips)
!rm -rf input_folder output_folder *.zip

print("✅ All cleaned up!")
print("\n📂 Current directory:")
!ls -la

✅ All cleaned up!

📂 Current directory:
total 16
drwxr-xr-x 1 root root 4096 May  2 20:30 .
drwxr-xr-x 1 root root 4096 May  2 19:14 ..
drwxr-xr-x 4 root root 4096 Apr 16 13:33 .config
drwxr-xr-x 1 root root 4096 Apr 16 13:33 sample_data


In [ ]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving C.zip to C.zip
Files extracted to 'input_folder/'


In [ ]:
# 3. Load model
model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-6.7b-instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-instruct")
tokenizer.pad_token = tokenizer.eos_token  # Add this line
tokenizer.padding_side = 'left'


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
ONE_SHOT_C = """\
#include <stdio.h>
#include <stdlib.h>
#define MAX 100
struct Node {
    int data ;
    struct Node * left , * right ;
} ;
struct Node * newNode ( int data ) {
    struct Node * node = ( struct Node * ) malloc ( sizeof ( struct Node ) ) ;
    node -> data = data ;
    node -> left = node -> right = NULL ;
    return node ;
}
int findMax ( int arr [ ] , int n ) {
    int max = arr [ 0 ] ;
    for ( int i = 1 ; i < n ; i ++ ) {
        if ( arr [ i ] > max )
            max = arr [ i ] ;
    }
    return max ;
}
int main ( ) {
    int arr [ ] = { 3 , 1 , 4 , 1 , 5 , 9 , 2 , 6 } ;
    int n = sizeof ( arr ) / sizeof ( arr [ 0 ] ) ;
    int result = findMax ( arr , n ) ;
    printf ( "Maximum value is %d\n" , result ) ;
    struct Node * root = newNode ( 1 ) ;
    root -> left = newNode ( 2 ) ;
    root -> right = newNode ( 3 ) ;
    getchar ( ) ;
    return 0 ;
}
"""

ONE_SHOT_CPP = """\
#include <iostream>
#include <cstdlib>
using namespace std ;
#define MAX 100
struct Node {
    int data ;
    struct Node * left , * right ;
} ;
struct Node * newNode ( int data ) {
    struct Node * node = ( struct Node * ) malloc ( sizeof ( struct Node ) ) ;
    node -> data = data ;
    node -> left = node -> right = nullptr ;
    return node ;
}
int findMax ( int arr [ ] , int n ) {
    int max = arr [ 0 ] ;
    for ( int i = 1 ; i < n ; i ++ ) {
        if ( arr [ i ] > max )
            max = arr [ i ] ;
    }
    return max ;
}
int main ( ) {
    int arr [ ] = { 3 , 1 , 4 , 1 , 5 , 9 , 2 , 6 } ;
    int n = sizeof ( arr ) / sizeof ( arr [ 0 ] ) ;
    int result = findMax ( arr , n ) ;
    cout << "Maximum value is " << result << "\n" ;
    struct Node * root = newNode ( 1 ) ;
    root -> left = newNode ( 2 ) ;
    root -> right = newNode ( 3 ) ;
    getchar ( ) ;
    return 0 ;
}
"""

def translate_batch(c_code_list):
    all_messages = []
    for c_code in c_code_list:

        system_prompt = """You are an expert code translator. Your ONLY task is to convert C code to C++ code.
Rules you MUST follow:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior."""

        user_prompt = f"""Translate this C code to C++ code.

Here is an example:

Input C Code:
{ONE_SHOT_C}

Expected C++ Output:
{ONE_SHOT_CPP}

Now translate this:

C Code:
{c_code}

C++ Code:
"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ]
        all_messages.append(messages)

    # Apply chat template to all messages
    prompt_texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True
        )
        for msgs in all_messages
    ]

    # Batch tokenization with padding
    inputs = tokenizer(
        prompt_texts,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # ── MEMORY CLEANUP ────────────────────────────
    torch.cuda.empty_cache()
    gc.collect()

    start = time.time()

    # Single batch generation
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.3
    )

    # ── TRACKING ──────────────────────────────────
    elapsed = time.time() - start
    print(f"  → Generation took: {elapsed:.1f} seconds")

    # Decode all outputs
    input_lengths = inputs['attention_mask'].sum(dim=1)

    results = []
    for i, output in enumerate(outputs):
        generated_ids = output[input_lengths[i]:]
        decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)
        decoded = decoded.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()
        results.append(decoded)

    return results

In [ ]:
import time
import gc

In [ ]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                print(f"\nProcessing batch — files: {batch_files}")  # add this
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")


                gc.collect()
                torch.cuda.empty_cache()
                # Clear batch lists
                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")


Processing batch — files: ['1971.c', '3318.c', '264.c', '2537.c']


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


  → Generation took: 19.7 seconds
Translated: input_folder/C/1971.c → output_folder/C/1971.cpp
Translated: input_folder/C/3318.c → output_folder/C/3318.cpp
Translated: input_folder/C/264.c → output_folder/C/264.cpp
Translated: input_folder/C/2537.c → output_folder/C/2537.cpp


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.



Processing batch — files: ['2202.c', '1699.c', '2065.c', '10643.c']
  → Generation took: 20.0 seconds
Translated: input_folder/C/2202.c → output_folder/C/2202.cpp
Translated: input_folder/C/1699.c → output_folder/C/1699.cpp
Translated: input_folder/C/2065.c → output_folder/C/2065.cpp
Translated: input_folder/C/10643.c → output_folder/C/10643.cpp


...


Processing batch — files: ['2588.c', '2305.c', '12748.c', '101.c']
  → Generation took: 19.6 seconds
Translated: input_folder/C/2588.c → output_folder/C/2588.cpp
Translated: input_folder/C/2305.c → output_folder/C/2305.cpp
Translated: input_folder/C/12748.c → output_folder/C/12748.cpp
Translated: input_folder/C/101.c → output_folder/C/101.cpp

Processing batch — files: ['7558.c', '1661.c', '7322.c', '1973.c']


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


  → Generation took: 19.6 seconds
Translated: input_folder/C/7558.c → output_folder/C/7558.cpp
Translated: input_folder/C/1661.c → output_folder/C/1661.cpp
Translated: input_folder/C/7322.c → output_folder/C/7322.cpp
Translated: input_folder/C/1973.c → output_folder/C/1973.cpp


Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


  → Generation took: 19.4 seconds
Translated: input_folder/C/10624.c → output_folder/C/10624.cpp
Translated: input_folder/C/935.c → output_folder/C/935.cpp


In [ ]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [ ]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r output.zip output_folder
colab_files.download('output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
  adding: output_folder/ (stored 0%)
  adding: output_folder/C/ (stored 0%)
  adding: output_folder/C/1706.cpp (deflated 48%)
  adding: output_folder/C/272.cpp (deflated 68%)
  adding: output_folder/C/4746.cpp (deflated 59%)
  adding: output_folder/C/10624.cpp (deflated 47%)
  adding: output_folder/C/134.cpp (deflated 43%)
  adding: output_folder/C/9298.cpp (deflated 51%)
  adding: output_folder/C/10813.cpp (deflated 59%)
  adding: output_folder/C/101.cpp (deflated 44%)
  adding: output_folder/C/1541.cpp (deflated 47%)
  adding: output_folder/C/2104.cpp (deflated 64%)
  adding: output_folder/C/12593.cpp (deflated 58%)
  adding: output_folder/C/7023.cpp (deflated 78%)
  adding: output_folder/C/2142.cpp (deflated 55%)
  adding: output_folder/C/7648.cpp (deflated 53%)
  adding: output_folder/C/10994.cpp (deflated 55%)
  adding: output_folder/C/848.cpp (deflated 52%)
  adding: output_folder/C/8793.cpp (deflated 54%)
  adding: output_folder/C/4768.cpp (deflated 49%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.


In [ ]:
print(model.generation_config)


GenerationConfig {
  "bos_token_id": 32013,
  "eos_token_id": 32021
}



In [ ]:
import os

count = 0
for root, _, files in os.walk('output_folder'):
    for file in files:
        if file.endswith('.cpp'):
            count += 1

print(f"Files translated so far: {count}")

Files translated so far: 402
